In [ ]:
# === Core ===
import numpy as np
import pandas as pd
import re
import ast
from datetime import datetime, timedelta
import warnings

# === Geospatial ===
import geopandas as gpd
from shapely.geometry import Point, shape

# === Visualization ===
import matplotlib.pyplot as plt

# === Modeling & Evaluation ===
from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import OneHotEncoder

# === Other Utilities ===
from IPython.display import display
import requests

# Suppress warnings
warnings.filterwarnings("ignore")

In [ ]:
# It was our idea to get neighborhood data from the city of Austin
# We found the website where it was available for download
# chatGPT helped write this code to put the json data into a dataframe

# Base OData API endpoint - where the data lives
base_url = "https://data.austintexas.gov/api/odata/v4/inrm-c3ee"

# initialize a place to collect everything
all_data = []
url = base_url

# fethes things one page at a time
while url:
    print(f"Fetching: {url}")
    response = requests.get(url)
    data = response.json()
    
    # adds the thing that it fetched by pulling out the "value"
    all_data.extend(data['value'])

    # check if there's a link to the next page
    url = data.get('@odata.nextLink', None)

# convert all results to a DataFrame
neighborhoods = pd.DataFrame(all_data)

In [ ]:
#### CHANGE WHICH DF YOU ARE CONVERTING HERE ####

NLP_df = housing_data_numeric_NLP

# NLP_df = housing_data_binary_NLP

In [ ]:
# chatGPT assist here to map lat / longs to neighborhoods and one-hot-encode

# load the data
houses_df = NLP_df

# create Point geometry from longitude and latitude (note order: lon, lat)
# each row's coords are put into a shapely Point file
# wraps the df in GeoDataFrame
# ensures coordinate system is WGS84
houses_df['geometry'] = [Point(lon, lat) for lon, lat in zip(houses_df['longitude'], houses_df['latitude'])]
houses_gdf = gpd.GeoDataFrame(houses_df, geometry='geometry', crs="EPSG:4326")  # WGS 84

# load and convert neighborhood data (from City of Austin code above)
neigh_df = neighborhoods

# convert each dict into a Shapely geometry
# had to go into that file and figure out the geometry was in "the_geom"
neigh_df['geometry'] = neigh_df['the_geom'].apply(shape)
neigh_gdf = gpd.GeoDataFrame(neigh_df, geometry='geometry', crs="EPSG:4326")

# spatial join to add planning_area_name
# planning area name is the neighborhood in the city of Austin code
# if the home is in the neighborhood, it gets a planning_area_name assigned
joined = gpd.sjoin(houses_gdf, neigh_gdf[['planning_area_name', 'geometry']], how='left', predicate='within')

# one-hot encode planning_area_name
# each neighborhood gets a dummy variable with the prefix hood_
joined_encoded = pd.get_dummies(joined, columns=['planning_area_name'], prefix = 'hood_')

# drop geometry and index_right (from spatial join)
# gets rid of unnecessary columns
joined_encoded = joined_encoded.drop(columns=['geometry', 'index_right','planning_area_name'], errors='ignore')

In [ ]:
#### continue here with joined_encoded ####